# Feed-forward parallelism sweep — N-MNIST MLP

Reproduces the experiment of Section 5.5.1 of the dissertation: a time-driven
percent-parallel sweep over the N-MNIST multilayer perceptron, measured against
an event-driven component generated from the same NIR graph.

**What it produces.** One synthesised design per sweep point, with latency,
post-synthesis resource use and a vectorless power estimate, plus the report
and the two figures that the results chapter uses.

**Prerequisites.** Vitis HLS and Vivado on the `PATH`, and a kernel whose
interpreter has the project dependencies. Everything below shells out to the
same scripts the command line uses, so the notebook and a terminal run produce
identical artefacts.

**Cost.** About five hours end to end on a 16 GB host, of which the last point
alone takes four. The pipeline step is written so that points already measured
are skipped, and so that a subset can be selected.

In [ ]:
import json, subprocess, sys
from pathlib import Path

AQUI = Path.cwd()
RAIZ = AQUI.parent
RUNS = RAIZ / "sim" / "runs"
PY = sys.executable          # o mesmo interpretador do kernel roda os scripts

def executar(args, **kw):
    """Roda um comando na raiz do repositorio e devolve o codigo de saida."""
    print("$", " ".join(str(a) for a in args))
    return subprocess.run(args, cwd=RAIZ, **kw).returncode

def ultimo_resumo(componente):
    """summary.json da run mais recente que produziu resultado, ou None."""
    achados = sorted((RUNS / componente).glob("*/reports/summary.json")) \
        if (RUNS / componente).is_dir() else []
    return json.loads(achados[-1].read_text()) if achados else None

print("repositorio:", RAIZ)
print("python     :", PY)
for f in ("vitis-run", "vivado"):
    r = subprocess.run(["which", f], capture_output=True, text=True)
    print(f"{f:11s}:", r.stdout.strip() or "NAO ENCONTRADO no PATH")

# --- configuracao do sweep, num lugar so ---------------------------------
UNIDADES = [1, 2, 4, 8, 16, 32, 64, 128, 256]   # elementos na camada dominante
FORCAR = False        # True regenera componentes que ja existem
SO_FALTANTES = True   # False refaz pontos que ja tem resultado
TIMEOUT_MIN = 300     # por componente; o maior ponto precisa de ~4 h


## The sweep

Each point requests `p = U / W` on the dominant dense layer, whose static work
domain is `W = 100 352` multiply-accumulate operations. Writing the request
this way makes the requested fraction and the effective one coincide there, and
names each point by the element count it is meant to resolve.

The generator refuses to overwrite an existing component unless `--force` is
given: the projects are pipeline inputs, and silently regenerating one would
invalidate the runs already recorded against its hash.

In [ ]:
cmd = [PY, "MLP_test/gerar_componentes_percent_parallelism.py",
       "--units", *map(str, UNIDADES)]
if FORCAR:
    cmd.append("--force")
executar(cmd)

## The resolved plan

`p` is model-wide, and every layer resolves its own element count from its own
work domain. The table below is what makes that concrete: the dominant layer
follows the request, while the smaller layers stay serial until `p` grows
enough to round them up. That is why the latency stops halving cleanly at the
last two points.

In [ ]:
print(f"{'componente':<34s} {'linear_0':>9s} {'lif_0':>6s} {'linear_1':>9s} {'lif_1':>6s}")
for u in UNIDADES:
    m = AQUI / f"hls_time_driven_percent_w{u:04d}" / "parallelism_manifest.json"
    if not m.is_file():
        continue
    camadas = json.loads(m.read_text())["layers"]
    u_por_camada = {c["name"]: c["processing_elements"] for c in camadas}
    print(f"{m.parent.name:<34s}" + "".join(
        f"{u_por_camada.get(n, '-'):>{w}}" for n, w in
        (("linear_0", 10), ("lif_0", 7), ("linear_1", 10), ("lif_1", 7))))

## Running the pipeline

Each component goes through project creation, C simulation, high-level
synthesis, IP export, out-of-context Vivado synthesis and a vectorless power
report. Components that already carry a usable `summary.json` are skipped, so
re-executing this cell is cheap and only fills gaps.

Two practical notes. The Vitis workspace under each run is by far the largest
artefact and nothing downstream reads it, so it is removed once a component
finishes — without that the sweep exhausts the disk. And the per-component
timeout is generous because the largest point needs two and a half hours in
high-level synthesis alone.

In [ ]:
import shutil


for u in UNIDADES:
    comp = f"hls_time_driven_percent_w{u:04d}"
    if SO_FALTANTES and ultimo_resumo(comp):
        print(f"{comp}: ja medido, pulando")
        continue
    rc = executar(["timeout", f"{TIMEOUT_MIN}m", PY, "-m", "sim", "run",
                   "--project", f"MLP_test/{comp}", "--to", "power"])
    for projeto in (RUNS / comp).glob("*/project"):
        shutil.rmtree(projeto, ignore_errors=True)
    print(f"{comp}: exit={rc}")

## Results

Latency is the cycles of one time step, taken from synthesis; resources are
Vivado out-of-context utilisation; energy multiplies the estimated average
power by the execution window. A point that did not complete shows as missing
rather than as a zero.

In [ ]:
W = 100_352
print(f"{'p':>10s} {'ciclos':>8s} {'LUT':>9s} {'FF':>9s} {'DSP':>6s} {'BRAM':>6s} "
      f"{'W':>6s} {'uJ/passo':>9s}")
for u in UNIDADES + [None]:
    comp = "hls_event_driven_scalar" if u is None else f"hls_time_driven_percent_w{u:04d}"
    s = ultimo_resumo(comp)
    if not s:
        print(f"{'ED' if u is None else u/W:>10} (sem resultado)")
        continue
    passos = (s.get("workload") or {}).get("total_logical_steps")
    cos = (s.get("cosim") or {}).get("total_execution_cycles")
    lat = cos / passos if cos and passos else float(s["hls"]["latency"]["worst_case"])
    r, pw = s["vivado_utilization"]["resources"], s["power"]
    t = pw["total_on_chip_power_w"]
    rot = "ED" if u is None else f"{u/W:.1e}"
    print(f"{rot:>10s} {lat:8,.0f} {r['lut']['used']:9,.0f} {r['ff']['used']:9,.0f} "
          f"{r['dsp']['used']:6,.0f} {r['bram']['used']:6.1f} {t:6.3f} "
          f"{t*lat/150e6*1e6:9.2f}")

## Report and figures

The report script is the single source of the numbers used in the dissertation:
it re-reads the run summaries, writes the markdown report, and renders the two
figures that appear as Figures 17 and 18. Running it after the sweep keeps
report, figures and chapter in agreement.

In [ ]:
executar([PY, "MLP_test/gerar_relatorio_percent_parallelism.py"])

from IPython.display import Image, Markdown, display
for nome in ("tempo_energia", "recursos"):
    display(Image(filename=str(AQUI / f"relatorio_percent_parallelism_{nome}.png")))
display(Markdown((AQUI / "relatorio_percent_parallelism.md").read_text()[:1500] + "\n\n..."))